In [1]:
pip install unsloth vllm

In [2]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
# Install latest Hugging Face for Llama-3.1
!pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0

In [3]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [4]:
from google.colab import files
uploaded = files.upload()


Saving simplification_final_dataset.csv to simplification_final_dataset.csv


In [5]:
from unsloth import FastModel
import torch

fourbit_models = [
    # LLaMA 3 models
    "unsloth/Llama-3.1-8B",  # Specifically for Llama-3.1-8B
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",

    # Other popular models!
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",  # Using Llama-3.1-8B model
    max_seq_length = 2048,                # Max sequence length for longer context
    load_in_4bit = True,                  # 4-bit quantization for reduced memory usage
    load_in_8bit = False,                 # Keep as False for better accuracy and memory use
    full_finetuning = False,              # If you need finetuning, set it True
    # token = "hf_...",                   # Use this token for gated models if needed
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-17 07:04:51 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.49.0. vLLM: 0.8.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

In [6]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [ ]:
filename = next(iter(uploaded))  # Get the first uploaded file's name

# ✅ Load Dataset
from datasets import load_dataset
dataset = load_dataset("csv", data_files=filename)

//split the dataset into train and test sets
train_test_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]


# ✅ Format dataset using Chat Template
def format_chat(example):
    return {
        "messages": [
            {"role": "user", "content": f"You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\n{example['legal_text']}"},
            {"role": "assistant", "content": f'Simplified Explanation: {example["simplified_text"]}'},
        ]
    }

dataset = dataset.map(format_chat)

In [32]:
dataset[1]['messages']




[{'content': 'You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\nOn the appointed day―\n(a) the Common and all other property which immediately before that\ndate was the property of the Churchwardens and was used or held in\nconnection with the Common; and\n(b) all rights and liabilities of the Churchwardens subsisting immediately\nbefore that date which were acquired or incurred in connection with\nthe Common, are transferred to and vest in the Trust free of any trusts established under the 1777 Act.',
  'role': 'user'},
 {'content': 'Simplified Explanation: Transfer of Property and Rights: On the specified date:\nThe property and all other assets that previously belonged to the Chur

In [33]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

In [34]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

In [35]:
dataset[100]

{'legal_text': 'Whilst the Common is in its ownership―\n(a) the Trust must remain a charity;\n(b) the objects of the Trust must include the primary objects.\nFrom the appointed day the Churchwardens shall not be liable for any act, event, failure to act or omission so far as the act, event, failure to act or omission relates to the Common and occurred before the appointed day.\nWhere the transfer and vesting of the Common or any part of the Common effected by subsection (1) is a registrable disposition under the Land Registration Act 2002, the Trust must apply to the Chief Land Registrar for registration in the register of title of a restriction to reflect section 12(2).',
 'simplified_text': "While the Common is owned by the Trust:\nThe Trust must continue to operate as a charity.\nThe Trust's mission must include the main objectives.\n\nFrom the appointed day, the Churchwardens will not be responsible for any actions, events, failures to act, or omissions related to the Common that o

In [44]:
def apply_chat_template(examples):
    texts = []
    for message in examples["messages"]:
        if isinstance(message, dict):
            text = message.get('content', '')  # Safely extract 'content' if it exists
        else:
            text = str(message)  # If it's not a dictionary, convert it to a string

        texts.append(text)

    return {"text": texts}
pass
dataset = dataset.map(apply_chat_template, batched = True)

Map:   0%|          | 0/2126 [00:00<?, ? examples/s]

In [46]:
dataset[100]["text"]


'[{\'content\': \'You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\\nWhilst the Common is in its ownership―\\n(a) the Trust must remain a charity;\\n(b) the objects of the Trust must include the primary objects.\\nFrom the appointed day the Churchwardens shall not be liable for any act, event, failure to act or omission so far as the act, event, failure to act or omission relates to the Common and occurred before the appointed day.\\nWhere the transfer and vesting of the Common or any part of the Common effected by subsection (1) is a registrable disposition under the Land Registration Act 2002, the Trust must apply to the Chief Land Registrar for registration in the register of titl

In [53]:
def convert_to_gemma_format(examples):
    conversations = []
    for conversation in examples["messages"]:
        formatted_conversation = ""

        for message in conversation:
            role = message["role"]
            content = message["content"]

            # Format the content based on role (user or assistant)
            if role == "user":
                formatted_conversation += f"<start_of_turn>user\n{content}<end_of_turn>\n"
            elif role == "assistant":
                formatted_conversation += f"<start_of_turn>model\n{content}<end_of_turn>\n"

        # Append the formatted conversation to the list
        conversations.append(formatted_conversation)

    return {"text": conversations}

# Applying the transformation function to your dataset
dataset = dataset.map(convert_to_gemma_format, batched=True)


Map:   0%|          | 0/2126 [00:00<?, ? examples/s]

In [54]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2126 [00:00<?, ? examples/s]

In [55]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=2):   0%|          | 0/2126 [00:00<?, ? examples/s]

In [56]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<|begin_of_text|><start_of_turn>user\nYou are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\nWhilst the Common is in its ownership―\n(a) the Trust must remain a charity;\n(b) the objects of the Trust must include the primary objects.\nFrom the appointed day the Churchwardens shall not be liable for any act, event, failure to act or omission so far as the act, event, failure to act or omission relates to the Common and occurred before the appointed day.\nWhere the transfer and vesting of the Common or any part of the Common effected by subsection (1) is a registrable disposition under the Land Registration Act 2002, the Trust must apply to the Chief Land Registrar for registration in the 

In [57]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

"                                                                                                                                                                                                                               Simplified Explanation: While the Common is owned by the Trust:\nThe Trust must continue to operate as a charity.\nThe Trust's mission must include the main objectives.\n\nFrom the appointed day, the Churchwardens will not be responsible for any actions, events, failures to act, or omissions related to the Common that occurred before the appointed day.\n\nIf the transfer of the Common or any part of it is a registrable transaction under the Land Registration Act 2002, the Trust must request the Chief Land Registrar to register a restriction in the title register to reflect section 12(2).<end_of_turn>\n"

In [58]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.67 GB of memory reserved.


In [59]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,126 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520/8,000,000,000 (0.26% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.773600
2,0.709300
3,0.829400
4,0.910900
5,0.792800
6,0.871400
7,0.786000
8,0.687700
9,0.662900
10,0.602100


In [73]:
local_directory = "C:/Users/Ideapad/OneDrive/Documents/Legal_NLP/Legal-Simplification-mrabbani-main/llama_legal_simplifier"
# Check if the directory exists
if not os.path.exists(local_directory):
    print(f"Directory {local_directory} does not exist.")
else:
    print(f"Directory {local_directory} exists.")


Directory C:/Users/Ideapad/OneDrive/Documents/Legal_NLP/Legal-Simplification-mrabbani-main/llama_legal_simplifier does not exist.


In [76]:
# Save the model and tokenizer to the local directory
model.save_pretrained(local_directory)
tokenizer.save_pretrained(local_directory)

print(f"Model and tokenizer saved to {local_directory}")

Model and tokenizer saved to C:/Users/Ideapad/OneDrive/Documents/Legal_NLP/Legal-Simplification-mrabbani-main/llama_legal_simplifier


In [77]:
import os

# List files in the directory to confirm the files are saved
print("Files in the directory:", os.listdir(local_directory))


Files in the directory: ['README.md', 'special_tokens_map.json', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json']


In [69]:
import gc
tensor = None
gc.collect()
torch.cuda.empty_cache()


In [70]:
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   5868 MiB |   6975 MiB |   9206 GiB |   9200 GiB |
|       from large pool |   5644 MiB |   6668 MiB |   9143 GiB |   9138 GiB |
|       from small pool |    224 MiB |    309 MiB |     62 GiB |     62 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   5868 MiB |   6975 MiB |   9206 GiB |   9200 GiB |
|       from large pool |   5644 MiB |   6668 MiB |   9143 GiB |

In [81]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

451.2837 seconds used for training.
7.52 minutes used for training.
Peak reserved memory = 11.475 GB.
Peak reserved memory for training = 5.805 GB.
Peak reserved memory % of max memory = 77.844 %.
Peak reserved memory for training % of max memory = 39.38 %.
